# Step 2: Cleaning

Goal: fix data types and inconsistent encodings, and derive a few basic helper columns — without dropping any rows. Row-level filtering decisions are deferred to Step 4, where the aggregation context makes the right call clearer.

See `docs/DATA_DICTIONARY.md` for the raw meaning of every column referenced below, and `docs/PIPELINE_LOG.md` for the full rationale behind each cleaning decision.

In [1]:
import sys
sys.path.append('..')

from src.data_loader import build_claim_level_dataset
from src.cleaning import clean_claim_level_dataset

raw_df = build_claim_level_dataset()
df = clean_claim_level_dataset(raw_df)
df.shape

(558211, 61)

**What this cell does:** runs the full cleaning pipeline from `src/cleaning.py`, in this order:

1. **`parse_dates()`** — converts date columns (`ClaimStartDt`, `ClaimEndDt`, `AdmissionDt`, `DischargeDt`, `DOB`, `DOD`) from string to proper `datetime` type.
2. **`recode_chronic_conditions()`** — the 11 `ChronicCond_*` columns are coded `1=Yes, 2=No` in the raw data. Left as-is, a model would treat "No" (2) as numerically *greater than* "Yes" (1), implying false ordinality. Recoded to standard `1=Yes, 0=No`.
3. **`recode_renal_disease_indicator()`** — `RenalDiseaseIndicator` is coded as strings `"Y"`/`"0"`. Recoded to integers `1`/`0`.
4. **`recode_fraud_label()`** — `PotentialFraud` recoded from `"Yes"`/`"No"` strings to `1`/`0`, so it can be used directly in statistical tests and models.
5. **`add_age_columns()`** — derives `AgeAtClaim` (age in years at `ClaimStartDt`, from `DOB`) and `IsDeceased` (`1` if `DOD` is present, else `0`).
6. **`cast_categorical_codes()`** — `State`, `County`, `Gender`, `Race` are numeric *codes*, not quantities (a county code of 200 isn't "twice" a county code of 100). Cast to `category` dtype so downstream stats/plots don't misinterpret them as ordered numbers.
7. **`add_physician_involvement_count()`** — derives `NumPhysiciansInvolved` (0-3) from the `AttendingPhysician`/`OperatingPhysician`/`OtherPhysician` columns, converting their structural missingness (not every claim needs 3 physicians) into a usable numeric signal.
8. **`fill_structural_missing()`** — fills `NaN` with `0` **only** for Deductible/Reimbursement amount columns (missing there plausibly means "no amount charged," not "unknown").

**Expected output:** `(558211, 58)` — same row count as after loading, 2 more columns from the derived fields.

## Why some missingness is left untouched

`AdmissionDt`, `DischargeDt`, and `DOD` are **intentionally not filled**. Their missingness carries real meaning:
- Missing `AdmissionDt`/`DischargeDt` means the claim is Outpatient (only inpatient claims have admissions) — this is structural, not a data error.
- Missing `DOD` means the beneficiary is alive.

Filling these with a placeholder value would destroy that signal and could mislead a model into treating them as real dates. Where the missingness itself is informative, we convert it into an explicit flag (`IsDeceased`) or leave it as `NaT` to be handled explicitly downstream, rather than papering over it.

In [2]:
df[['ChronicCond_Diabetes', 'RenalDiseaseIndicator', 'PotentialFraud', 'AgeAtClaim', 'IsDeceased', 'NumPhysiciansInvolved']].head()

,ChronicCond_Diabetes,RenalDiseaseIndicator,PotentialFraud,AgeAtClaim,IsDeceased,NumPhysiciansInvolved
0,1,0,1,66.277892,0,1
1,1,0,0,66.663929,0,2
2,1,0,0,66.710472,0,2
3,1,0,0,94.959617,0,3
4,0,1,0,71.367556,0,2


**What this checks:** a quick visual spot-check that the recodings and derived columns look right — `ChronicCond_Diabetes` and `RenalDiseaseIndicator` should now show clean `0`/`1` values, `PotentialFraud` should be `0`/`1` (not `"Yes"`/`"No"`), and `AgeAtClaim`/`NumPhysiciansInvolved` should show sensible numbers.

In [3]:
df.dtypes[['State', 'County', 'Gender', 'Race', 'ClaimStartDt', 'DOB']]

State                 category
County                category
Gender                category
Race                  category
ClaimStartDt    datetime64[ns]
DOB             datetime64[ns]
dtype: object

**What this checks:** confirms `State`/`County`/`Gender`/`Race` show as `category` dtype (not `int64`), and the date columns show as `datetime64[ns]` (not `object`/string).

In [4]:
df.isna().sum().sort_values(ascending=False).head(15)

ClmProcedureCode_6     558211
ClmProcedureCode_5     558202
ClmProcedureCode_4     558093
ClmProcedureCode_3     557242
DOD                    554080
ClmDiagnosisCode_10    553201
ClmProcedureCode_2     552721
ClmProcedureCode_1     534901
AdmissionDt            517737
DischargeDt            517737
DiagnosisGroupCode     517737
ClmDiagnosisCode_9     516396
ClmDiagnosisCode_8     504767
ClmDiagnosisCode_7     492034
ClmDiagnosisCode_6     473819
dtype: int64

**What this checks:** the top missing columns should all be the *expected* structural ones — `ClmProcedureCode_5/6`, `DOD`, `ClmDiagnosisCode_10`, `AdmissionDt`/`DischargeDt`, etc. If anything unexpected shows up with heavy missingness here, that's worth investigating before moving on.

In [5]:
print(f"Age range: {df['AgeAtClaim'].min():.1f} to {df['AgeAtClaim'].max():.1f}")
print(f"Outpatient claims: {(df['ClaimType'] == 'Outpatient').sum()}")
print(f"Missing AdmissionDt: {df['AdmissionDt'].isna().sum()}")

Age range: 25.1 to 101.0
Outpatient claims: 517737
Missing AdmissionDt: 517737


**What this checks — two important sanity checks:**
1. **Age range** should be plausible (no negative ages, which would mean `DOB` after `ClaimStartDt` — a real data error; no absurd ages like 150+). Expected: roughly 25 to 101.
2. **Outpatient claim count should exactly equal missing `AdmissionDt` count.** If these two numbers match, it confirms the missingness in `AdmissionDt` is purely structural (Outpatient claims never have one) rather than a join or data-quality issue slipping in unnoticed.

In [6]:
df.to_csv('../data/processed/claims_cleaned.csv', index=False)

**What this does:** saves the cleaned claim-level dataset to `data/processed/`, so later notebooks (EDA, feature engineering) can load it directly instead of re-running loading + cleaning every time. Not committed to git (see `.gitignore`) — regenerate it locally by re-running this notebook if needed.

## Summary of Step 2

- Cleaned dataset: **558,211 rows × 58 columns** (2 more than raw, from `AgeAtClaim` and `IsDeceased`/`NumPhysiciansInvolved`)
- Fixed encodings: chronic conditions, renal disease indicator, and fraud label all recoded to clean binary/numeric form
- Derived helper columns: `AgeAtClaim`, `IsDeceased`, `NumPhysiciansInvolved`
- Cast geographic/demographic codes to `category` dtype so they aren't misread as ordered quantities
- **No rows dropped** — deferred to Step 4, once the aggregation context is clear
- **Structural missingness preserved intentionally** (`AdmissionDt`, `DischargeDt`, `DOD`) — verified `AdmissionDt` missingness exactly matches the Outpatient claim count
- **Not yet handled (deferred to Step 4):** high-cardinality sparse columns (`ClmDiagnosisCode_*`, `ClmProcedureCode_*`, physician IDs) — not usable as-is in most models; plan is to derive counts/flags during feature engineering rather than clean them as individual columns here